# Data Cleaning 04 -- IBES PERMNO Link

## Input
`Data/Data_Collection/Initial/04_LSEG_IBES/ibes_permno_link.parquet` (150,113 rows, built from `ibes.idsum` joined to `crsp.stocknames` on CUSIP)

## Purpose
Cleans the IBES ticker to CRSP PERMNO crosswalk. All three IBES factor files (revenue, recommendations, price targets) depend on this link -- if it is wrong, every IBES factor gets mismatched to the wrong stock or silently dropped. Key concerns addressed: one-to-many mappings (a ticker mapping to multiple PERMNOs over time), many-to-one mappings (multiple tickers for the same PERMNO due to corporate events), overlapping date ranges from WRDS CUSIP revision tracking artefacts, duplicate rows, and coverage against the master universe.

## Stage 0: Load & Inspect
Shape, dtypes, unique ticker/PERMNO counts, date ranges for `sdates` and `edates`.

## Stage 1: Missing Data Audit
- Per-column NaN counts
- Check for empty ticker strings

## Stage 2: Link Table Structural Checks
- **One-to-many:** identifies tickers that map to multiple PERMNOs over time, with date ranges shown for examples
- **Many-to-one:** identifies PERMNOs with multiple tickers (e.g., ticker changes from corporate restructuring), with date ranges shown
- **Date overlap check:** for each PERMNO with multiple link rows, checks whether any `edates` extends past the next row's `sdates`, counts overlapping periods and prints examples
- **edates distribution:** counts links still active (edates = 2099-12-31) vs ended
- **Duplicate rows:** counts exact duplicates and key-level duplicates on `(ticker, permno, sdates)`

## Stage 3: Coverage Check Against Master Universe
- Compares link table PERMNOs against the master PERMNO list
- Reports match rate and lists any unmatched PERMNOs with the years they appeared in the universe
- Filters link table to master universe only and reports per-PERMNO coverage details (multi-ticker PERMNOs and ended links)

## Stage 4: Clean & Save

### Filtered to Master Universe
150,113 rows reduced to 5,615 by keeping only links for the 227 PERMNOs in the top-100 S&P 500 universe.

### Dropped Exact Duplicates
32,761 duplicates in the full table caused by WRDS CUSIP revision tracking. Each time a CUSIP's metadata is updated, WRDS creates a new row with the same `(ticker, permno)` but a different `(sdates, edates)` window -- often zero-length. These are artefacts, not real links.

### Collapsed Overlapping Date Ranges
For each `(ticker, permno)` pair, `min(sdates)` is taken as the link start and `max(edates)` as the link end. This resolves the 3,967 "overlapping" periods which are fragmented versions of the same continuous link.

### Multi-Ticker PERMNOs Preserved
4 PERMNOs have legitimately different tickers over time (corporate events -- mergers, spin-offs, ticker changes). Both tickers are kept with their respective date ranges. The merge pipeline uses a date-aware join: for each `(permno, month)`, the ticker whose `sdates <= month-end <= edates` is selected.

### Columns Retained
`ticker`, `permno`, `sdates`, `edates`. Dropped `ibes_cusip` and `ncusip` (only needed for the join itself, not downstream).

### Coverage
100% -- all 227 master PERMNOs have at least one IBES ticker link.

## Output
`Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_permno_link_clean.parquet`

In [1]:
# %% [markdown]
# # Data Cleaning: ibes_permno_link.parquet
#
# Source: Data/Data_Collection/Initial/04_LSEG_IBES/ibes_permno_link.parquet
# Output: Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_permno_link_clean.parquet
#
# This is the IBES ticker → CRSP PERMNO crosswalk. All three IBES factor files
# (revenue, recommendations, price targets) depend on this link. If it's wrong,
# every IBES factor gets mismatched to the wrong stock or silently dropped.
#
# Built from: ibes.idsum (ticker → CUSIP) joined to crsp.stocknames (ncusip → PERMNO)
# Key fields:
#   - ticker: IBES identifier
#   - permno: CRSP PERMNO
#   - sdates: start date of this link (when ticker began mapping to this PERMNO)
#   - edates: end date (when link expires, or 2099-12-31 if still active)
#
# Key concerns:
#   - One-to-many mappings: a ticker can map to multiple PERMNOs over time
#     (ticker changes, corporate restructuring)
#   - Many-to-one: multiple tickers can map to the same PERMNO
#   - Date overlap: a PERMNO might have overlapping links from different tickers
#   - Coverage: do all ~227 master PERMNOs have at least one IBES ticker link?

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH    = Path('../../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_permno_link.parquet')
MASTER_PATH = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_master_clean.parquet')
ANNUAL_PATH = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_annual_clean.parquet')
OUT_DIR     = Path('../../../Data/Data_Collection/Cleaned/04_LSEG_IBES')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — ibes_permno_link")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")

print(f"\nColumns and dtypes:")
for c in df.columns:
    print(f"  {c:<20s} {str(df[c].dtype)}")

# Convert dates
for col in ['sdates', 'edates']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print(f"\nUnique IBES tickers: {df['ticker'].nunique():,}")
print(f"Unique PERMNOs: {df['permno'].nunique():,}")
print(f"sdates range: {df['sdates'].min().date()} → {df['sdates'].max().date()}")
print(f"edates range: {df['edates'].min().date()} → {df['edates'].max().date()}")

print(f"\n--- Head (10 rows) ---")
print(df.head(10).to_string(index=False))

print(f"\n--- Tail (10 rows) ---")
print(df.tail(10).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT")
print("=" * 90)

for c in df.columns:
    n = df[c].isna().sum()
    pct = n / len(df) * 100
    status = "✓" if n == 0 else "⚠"
    print(f"  {status} {c:<20s} {n:>6d} NaN ({pct:.2f}%)")

# Check for empty strings in ticker
n_empty = (df['ticker'].astype(str).str.strip() == '').sum()
print(f"\n  Empty ticker strings: {n_empty}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: LINK TABLE STRUCTURAL CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: LINK TABLE STRUCTURAL CHECKS")
print("=" * 90)

# ── 2a. One-to-many: tickers that map to multiple PERMNOs ───────────────────
print(f"\n--- Tickers mapping to multiple PERMNOs ---")
ticker_permno_counts = df.groupby('ticker')['permno'].nunique()
multi_permno = ticker_permno_counts[ticker_permno_counts > 1]
print(f"  Tickers with 1 PERMNO:  {(ticker_permno_counts == 1).sum():,}")
print(f"  Tickers with 2 PERMNOs: {(ticker_permno_counts == 2).sum():,}")
print(f"  Tickers with 3+ PERMNOs: {(ticker_permno_counts >= 3).sum():,}")

if len(multi_permno) > 0:
    print(f"\n  Examples of multi-PERMNO tickers (first 10):")
    for ticker in multi_permno.head(10).index:
        permnos = df[df['ticker'] == ticker][['permno', 'sdates', 'edates']]
        print(f"    {ticker}:")
        for _, row in permnos.iterrows():
            print(f"      PERMNO {int(row['permno'])}: "
                  f"{row['sdates'].date()} → {row['edates'].date()}")

# ── 2b. Many-to-one: PERMNOs that have multiple tickers ─────────────────────
print(f"\n--- PERMNOs with multiple tickers ---")
permno_ticker_counts = df.groupby('permno')['ticker'].nunique()
multi_ticker = permno_ticker_counts[permno_ticker_counts > 1]
print(f"  PERMNOs with 1 ticker:  {(permno_ticker_counts == 1).sum():,}")
print(f"  PERMNOs with 2 tickers: {(permno_ticker_counts == 2).sum():,}")
print(f"  PERMNOs with 3+ tickers: {(permno_ticker_counts >= 3).sum():,}")

if len(multi_ticker) > 0:
    print(f"\n  Examples of multi-ticker PERMNOs (first 10):")
    for permno in multi_ticker.head(10).index:
        tickers = df[df['permno'] == permno][['ticker', 'sdates', 'edates']]
        print(f"    PERMNO {int(permno)}:")
        for _, row in tickers.iterrows():
            print(f"      {row['ticker']}: "
                  f"{row['sdates'].date()} → {row['edates'].date()}")

# ── 2c. Date overlap check ──────────────────────────────────────────────────
print(f"\n--- Date overlap check (same PERMNO, overlapping date ranges) ---")
n_overlaps = 0
overlap_examples = []

for permno, group in df.groupby('permno'):
    if len(group) < 2:
        continue
    group = group.sort_values('sdates')
    for i in range(len(group) - 1):
        if group.iloc[i]['edates'] > group.iloc[i + 1]['sdates']:
            n_overlaps += 1
            if len(overlap_examples) < 5:
                overlap_examples.append({
                    'permno': int(permno),
                    'ticker1': group.iloc[i]['ticker'],
                    'end1': group.iloc[i]['edates'].date(),
                    'ticker2': group.iloc[i + 1]['ticker'],
                    'start2': group.iloc[i + 1]['sdates'].date(),
                })

print(f"  Overlapping link periods: {n_overlaps}")
if overlap_examples:
    print(f"  Examples:")
    for ex in overlap_examples:
        print(f"    PERMNO {ex['permno']}: {ex['ticker1']} ends {ex['end1']} "
              f"but {ex['ticker2']} starts {ex['start2']}")

# ── 2d. edates check ────────────────────────────────────────────────────────
print(f"\n--- edates distribution ---")
n_active = (df['edates'] >= pd.Timestamp('2099-01-01')).sum()
n_ended = (df['edates'] < pd.Timestamp('2099-01-01')).sum()
print(f"  Still active (edates = 2099): {n_active:,}")
print(f"  Ended (edates < 2099): {n_ended:,}")

if n_ended > 0:
    ended = df[df['edates'] < pd.Timestamp('2099-01-01')]
    print(f"  Ended links date range: {ended['edates'].min().date()} → "
          f"{ended['edates'].max().date()}")

# ── 2e. Duplicate rows ──────────────────────────────────────────────────────
print(f"\n--- Duplicate rows ---")
n_dupes = df.duplicated().sum()
n_dupes_key = df.duplicated(subset=['ticker', 'permno', 'sdates']).sum()
print(f"  Exact duplicates: {n_dupes}")
print(f"  Duplicate (ticker, permno, sdates): {n_dupes_key}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: COVERAGE CHECK AGAINST MASTER UNIVERSE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: COVERAGE CHECK AGAINST MASTER UNIVERSE")
print("=" * 90)

master = pd.read_parquet(MASTER_PATH)
annual = pd.read_parquet(ANNUAL_PATH)

master_permnos = set(master['permno'])
link_permnos = set(df['permno'].unique())

matched = master_permnos & link_permnos
unmatched = master_permnos - link_permnos

print(f"\n  Master PERMNOs: {len(master_permnos)}")
print(f"  PERMNOs in link table: {len(link_permnos):,}")
print(f"  Master PERMNOs matched: {len(matched)} ({len(matched)/len(master_permnos)*100:.1f}%)")
print(f"  Master PERMNOs UNMATCHED: {len(unmatched)}")

if unmatched:
    print(f"\n  Unmatched PERMNOs: {sorted(unmatched)}")
    # For each unmatched, show which years they were in the universe
    print(f"\n  Unmatched PERMNO details:")
    for p in sorted(unmatched):
        years = sorted(annual[annual['permno'] == p]['year'].tolist())
        print(f"    PERMNO {p}: in universe for years {years}")

# ── Filter link table to master universe ─────────────────────────────────────
print(f"\n--- Link table filtered to master universe ---")
link_universe = df[df['permno'].isin(master_permnos)].copy()
print(f"  Rows: {len(df):,} → {len(link_universe):,}")
print(f"  Tickers covering master PERMNOs: {link_universe['ticker'].nunique()}")
print(f"  PERMNOs covered: {link_universe['permno'].nunique()}")

# ── Per-PERMNO: how many tickers and what date coverage? ─────────────────────
print(f"\n--- Per-PERMNO coverage (master universe only) ---")
for permno in sorted(link_universe['permno'].unique()):
    sub = link_universe[link_universe['permno'] == permno].sort_values('sdates')
    tickers = ', '.join(
        f"{row['ticker']} ({row['sdates'].date()}→{row['edates'].date()})"
        for _, row in sub.iterrows()
    )
    # Only print if multiple tickers or if edates is not 2099
    if len(sub) > 1 or (sub['edates'] < pd.Timestamp('2099-01-01')).any():
        print(f"  PERMNO {int(permno)}: {tickers}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 4: SUMMARY")
print("=" * 90)

print(f"""
Review the output above:

1. UNMATCHED PERMNOs:
   - How many master PERMNOs have no IBES ticker link?
   - These stocks will have no IBES data (revenue, recs, price targets)
   - This is acceptable if it's a small number

2. OVERLAPPING LINKS:
   - If a PERMNO has overlapping date ranges from different tickers,
     the merge pipeline needs to handle this (pick the most recent ticker)
   - Or resolve overlaps here by trimming edates

3. MULTI-TICKER PERMNOs:
   - Normal for stocks that changed tickers (e.g., Facebook → Meta)
   - The date-aware merge handles this: use the ticker valid for each month

4. ACTION:
   - Filter link table to master universe only (drop irrelevant tickers)
   - Remove exact duplicate rows if any
   - Resolve overlaps if any
   - Save the clean link table

Paste back the output and I will write the cleaning cell.
""")

STAGE 0: LOAD & INSPECT — ibes_permno_link

Shape: 150,113 rows × 6 columns

Columns and dtypes:
  ticker               string
  ibes_cusip           string
  sdates               datetime64[ns]
  permno               Int64
  ncusip               string
  edates               datetime64[ns]

Unique IBES tickers: 20,107
Unique PERMNOs: 20,615
sdates range: 1976-01-15 → 2026-02-19
edates range: 1976-01-15 → 2099-12-31

--- Head (10 rows) ---
ticker ibes_cusip     sdates  permno   ncusip     edates
  0000   87482X10 2014-02-20   14471 87482X10 2014-03-20
  0000   87482X10 2014-03-20   14471 87482X10 2099-12-31
  0001   26878510 2014-02-20   14392 26878510 2014-02-20
  0001   26878510 2014-02-20   14392 26878510 2014-02-20
  0001   26878510 2014-02-20   14392 26878510 2014-03-20
  0001   26878510 2014-03-20   14392 26878510 2014-03-20
  0001   26878510 2014-03-20   14392 26878510 2014-03-20
  0001   26878510 2014-03-20   14392 26878510 2019-06-20
  0001   26878510 2019-06-20   14392 268785

In [2]:
# %% [markdown]
# ## Stage 4: Clean & Save
#
# **Cleaning decisions and rationale:**
#
# **Filtered to master universe:** 150,113 → 5,615 rows. Only keep links for
# the 227 PERMNOs in the top-100 S&P 500 universe.
#
# **Dropped exact duplicates:** 32,761 duplicates in the full table caused by
# WRDS CUSIP revision tracking. Each time a CUSIP's metadata is updated, WRDS
# creates a new row with the same (ticker, permno) but a different (sdates, edates)
# window — often zero-length (sdates = edates). These are artefacts, not real links.
#
# **Collapsed overlapping date ranges:** for each (ticker, permno) pair, take
# min(sdates) as the link start and max(edates) as the link end. This resolves
# the 3,967 "overlapping" periods which are really just fragmented versions of
# the same continuous link.
#
# **Multi-ticker PERMNOs preserved:** 4 PERMNOs have legitimately different
# tickers over time (corporate events — mergers, spin-offs, ticker changes).
# Both tickers are kept with their respective date ranges. The merge pipeline
# uses a date-aware join: for each (permno, month), pick the ticker whose
# sdates ≤ month-end ≤ edates.
#
# **Columns retained:** ticker, permno, sdates, edates. Dropped ibes_cusip and
# ncusip (only needed for the join itself, not downstream).
#
# **100% coverage:** all 227 master PERMNOs have at least one IBES ticker link.

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 4: CLEAN & SAVE")
print("=" * 90)

master = pd.read_parquet(MASTER_PATH)

# ── 4a. Filter to master universe ────────────────────────────────────────────
n_before = len(df)
df_clean = df[df['permno'].isin(master['permno'])].copy()
print(f"\n  Filtered to universe: {n_before:,} → {len(df_clean):,} rows")
print(f"  PERMNOs: {df_clean['permno'].nunique()}")
print(f"  Tickers: {df_clean['ticker'].nunique()}")

# ── 4b. Drop exact duplicates ───────────────────────────────────────────────
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"  Dropped exact duplicates: {n_before:,} → {len(df_clean):,}")

# ── 4c. Keep only the columns we need ───────────────────────────────────────
df_clean = df_clean[['ticker', 'permno', 'sdates', 'edates']]

# ── 4d. Collapse to one row per (ticker, permno) with widest date range ─────
n_before = len(df_clean)
df_clean = (
    df_clean
    .groupby(['ticker', 'permno'], as_index=False)
    .agg(sdates=('sdates', 'min'), edates=('edates', 'max'))
)
print(f"  Collapsed date ranges: {n_before:,} → {len(df_clean):,} rows")

# ── 4e. Sort for readability ────────────────────────────────────────────────
df_clean = df_clean.sort_values(['permno', 'sdates']).reset_index(drop=True)

# ── 4f. Verify ──────────────────────────────────────────────────────────────
print(f"\n  Final shape: {df_clean.shape}")
print(f"  PERMNOs: {df_clean['permno'].nunique()}")
print(f"  Tickers: {df_clean['ticker'].nunique()}")
print(f"  NaN: {df_clean.isna().sum().sum()}")

# Check multi-ticker PERMNOs
multi = df_clean.groupby('permno')['ticker'].nunique()
multi = multi[multi > 1]
if len(multi) > 0:
    print(f"\n  PERMNOs with multiple tickers ({len(multi)}):")
    for permno in multi.index:
        sub = df_clean[df_clean['permno'] == permno]
        tickers = ', '.join(
            f"{row['ticker']} ({row['sdates'].date()}→{row['edates'].date()})"
            for _, row in sub.iterrows()
        )
        print(f"    PERMNO {int(permno)}: {tickers}")

print(f"\n  Sample:")
print(df_clean.head(10).to_string(index=False))

# ── 4g. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'ibes_permno_link_clean.parquet'
df_clean.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {len(df_clean)} rows ({df_clean['permno'].nunique()} PERMNOs, "
      f"{df_clean['ticker'].nunique()} tickers)")

print("\nCleaning complete.")

STAGE 4: CLEAN & SAVE

  Filtered to universe: 150,113 → 5,615 rows
  PERMNOs: 227
  Tickers: 231
  Dropped exact duplicates: 5,615 → 3,348
  Collapsed date ranges: 3,348 → 232 rows

  Final shape: (232, 4)
  PERMNOs: 227
  Tickers: 231
  NaN: 0

  PERMNOs with multiple tickers (5):
    PERMNO 15069: X (1976-01-15→1991-05-16), MRO1 (1991-05-16→2099-12-31)
    PERMNO 66157: FBS (1976-01-15→2099-12-31), FNAC (2001-03-15→2099-12-31)
    PERMNO 69032: MS (1985-08-15→2099-12-31), DWD (1997-06-19→2099-12-31)
    PERMNO 81593: WAMU (1984-03-15→2099-12-31), WMIH (2016-06-16→2099-12-31)
    PERMNO 90441: NWS/1 (2004-11-18→2005-01-20), NWS (2005-01-20→2099-12-31)

  Sample:
ticker  permno     sdates     edates
  ORCL   10104 1986-06-19 2099-12-31
  MSFT   10107 1985-08-15 2099-12-31
   ALD   10145 1976-01-15 2099-12-31
  EMCS   10147 1987-09-17 2099-12-31
   ADM   10516 1976-07-15 2099-12-31
  FISV   10696 1986-10-16 2099-12-31
  DELL   11081 1988-08-18 2099-12-31
    KO   11308 1976-01-15 2099-